In [ ]:
%pip install -q sagemaker boto3 pandas numpy matplotlib

# Week 3 Monday -- Advanced SageMaker Architecture, Cost Optimization, and Security

On Friday you deployed models across four inference patterns and set up monitoring to catch drift. But someone has to pay the AWS bill. Today we learn how to slash training costs by up to 90% with Spot instances, right-size inference endpoints, set up auto-scaling, and lock down the environment with VPC isolation and KMS encryption. This is the operations and security layer that makes ML production-ready.

By the end of this notebook you will have:

1. **Configured** Managed Spot Training with checkpointing to reduce training costs by up to 90%.
2. **Combined** spot instances with HPO for compound savings.
3. **Configured** auto-scaling policies (target tracking and step scaling) for inference endpoints.
4. **Used** Inference Recommender to benchmark instance selection.
5. **Created** a CloudWatch dashboard with endpoint metrics.
6. **Configured** VPC, KMS, and IAM security controls for SageMaker workloads.
7. **Reviewed** the full ML lifecycle from Prepare to Monitor.

| Block | Content | Minutes |
|-------|---------|--------|
| Stage 1 | Cost Optimization for Training: Spot Instances, Checkpointing, HPO + Spot, Instance Right-Sizing | 55 |
| Stage 2 | Cost Optimization for Inference: Auto-Scaling, Inference Recommender, CloudWatch, Cost Comparison | 55 |
| Stage 3 | Security and Governance: VPC, PrivateLink, KMS, IAM, Full Lifecycle Review | 55 |

**Pairing callouts:**

- **Friday (W2):** Friday deployed models with serverless, async, batch transform, and multi-model patterns, then set up monitoring with data capture and baselines. Today we optimize those deployment patterns for cost and secure the entire pipeline.
- **Thursday (W2):** Thursday trained XGBoost with HPO. Today we show how to run those same HPO jobs with spot instances for compound savings.
- **Tuesday (W3):** Tomorrow introduces Amazon Bedrock and foundation models -- a different paradigm where you leverage pre-trained models instead of training from scratch.

## Setup

Run the cell below to establish your SageMaker session. This connects to your execution role, default S3 bucket, and region.

In [ ]:
import boto3
import sagemaker
from datetime import datetime

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
bucket = session.default_bucket()
prefix = "fraudshield"

sm_client = boto3.client("sagemaker")
s3 = boto3.client("s3")

TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

print(f"Region:  {region}")
print(f"Bucket:  s3://{bucket}")
print(f"Role:    ...{role[-30:]}")

In [ ]:
import json
import time
import numpy as np
import pandas as pd

runtime_client = boto3.client("sagemaker-runtime")

---
# STAGE 1 -- Cost Optimization for Training

**Connecting to Friday:** Friday you deployed models across four inference patterns and set up monitoring. Those deployments run on instances that cost money. Today we learn how to reduce that cost -- starting with training.

**Connecting to Thursday:** Thursday you ran HPO with multiple training jobs. Each job was an on-demand instance at full price. Today we show how to run those same jobs with spot instances for up to 90% savings.

## Managed Spot Training Overview

**What are spot instances?**

Spot instances are unused EC2 capacity that AWS sells at a steep discount -- up to 90% off on-demand pricing. The trade-off: AWS can reclaim the instance with a two-minute warning when it needs the capacity back.

For web servers, that is risky. For ML training, it is manageable -- because training jobs can **checkpoint** their progress and **resume** from where they left off.

**SageMaker Managed Spot Training** handles the spot lifecycle automatically:
- Requests spot capacity
- Starts training
- If interrupted, waits for new spot capacity
- Resumes training from the latest checkpoint
- Reports cost savings in the job output

**Key parameters:**

| Parameter | Purpose | Typical Value |
|-----------|---------|---------------|
| `use_spot_instances` | Enable spot training | `True` |
| `max_run` | Maximum seconds the training job can run | `1800` (30 min) |
| `max_wait` | Maximum seconds to wait for spot capacity (includes `max_run`) | `3600` (1 hr) |
| `checkpoint_s3_uri` | S3 path for saving checkpoints | `s3://<bucket>/checkpoints/` |

`max_wait` must be greater than or equal to `max_run`. The difference (`max_wait - max_run`) is the maximum time SageMaker will wait for spot capacity to become available.

**Savings potential:**
- Typical savings: 60-90% depending on instance type and availability
- A 4-hour training run on ml.p3.8xlarge at $14.68/hr saves ~$40 per run at 70% discount
- Savings compound across weekly retraining and HPO experiments

> **Discussion:** Why must `max_wait` be at least as large as `max_run`? What happens if spot capacity is unavailable for the entire `max_wait` period?

## STEP 1 -- Configure Spot Training with XGBoost

We configure a spot training job for the FraudShield XGBoost model. Three parameters change from a standard Estimator:
- `use_spot_instances=True`
- `max_wait=3600` (1 hour total)
- `max_run=1800` (30 minutes of actual training)
- `checkpoint_s3_uri` pointing to S3 for fault tolerance

> **What is happening:** We first regenerate the training data if needed, then create an XGBoost Estimator with spot training enabled. The checkpoint configuration ensures that if the spot instance is interrupted, training can resume from the last saved state.

In [ ]:
train_exists = False
val_exists = False
paginator = s3.get_paginator("list_objects_v2")
try:
    s3.head_object(Bucket=bucket, Key=f"{prefix}/data/xgb-train/train.csv")
    train_exists = True
except Exception:
    pass
try:
    s3.head_object(Bucket=bucket, Key=f"{prefix}/data/xgb-validation/validation.csv")
    val_exists = True
except Exception:
    pass

if not train_exists or not val_exists:
    np.random.seed(42)
    n = 2000
    data = pd.DataFrame({
        "amount": np.random.exponential(500, n).round(2),
        "hour": np.random.randint(0, 24, n),
        "distance_from_home": np.random.exponential(50, n).round(2),
        "transaction_count_24h": np.random.poisson(5, n),
        "is_international": np.random.binomial(1, 0.1, n),
        "merchant_risk_score": np.random.uniform(0, 1, n).round(3),
    })
    data["target"] = (
        (data["amount"] > 800) & (data["hour"] < 6) | (data["merchant_risk_score"] > 0.85)
    ).astype(int)
    noise = np.random.random(n) < 0.08
    data["target"] = (data["target"] ^ noise.astype(int))

    train = data.iloc[:1600]
    val = data.iloc[1600:]

    train_xgb = pd.concat([train["target"], train.drop("target", axis=1)], axis=1)
    val_xgb = pd.concat([val["target"], val.drop("target", axis=1)], axis=1)
    train_xgb.to_csv("train_xgb.csv", index=False, header=False)
    val_xgb.to_csv("val_xgb.csv", index=False, header=False)

    train.to_csv("train.csv", index=False)
    val.to_csv("validation.csv", index=False)

    s3.upload_file("train.csv", bucket, f"{prefix}/data/train/train.csv")
    s3.upload_file("validation.csv", bucket, f"{prefix}/data/validation/validation.csv")
    s3.upload_file("train_xgb.csv", bucket, f"{prefix}/data/xgb-train/train.csv")
    s3.upload_file("val_xgb.csv", bucket, f"{prefix}/data/xgb-validation/validation.csv")
    print("Training and validation data uploaded to S3.")
else:
    print("Training and validation data already present in S3.")

print(f"Train data: s3://{bucket}/{prefix}/data/xgb-train/train.csv")
print(f"Val data:   s3://{bucket}/{prefix}/data/xgb-validation/validation.csv")

In [ ]:
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput

xgb_image = image_uris.retrieve("xgboost", region, version="1.7-1")

checkpoint_s3_uri = f"s3://{bucket}/{prefix}/checkpoints/spot-training-{TIMESTAMP}"

spot_estimator = sagemaker.estimator.Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/spot-output/",
    sagemaker_session=session,
    use_spot_instances=True,
    max_wait=3600,
    max_run=1800,
    checkpoint_s3_uri=checkpoint_s3_uri,
)

spot_estimator.set_hyperparameters(
    objective="binary:logistic",
    num_round=100,
    max_depth=5,
    eta=0.2,
    eval_metric="auc",
)

print("Spot training estimator configured:")
print(f"  use_spot_instances: True")
print(f"  max_run:           1800s (30 min)")
print(f"  max_wait:          3600s (1 hr)")
print(f"  checkpoint_s3_uri: {checkpoint_s3_uri}")
print(f"  instance_type:     ml.m5.xlarge")
print(f"  image:             {xgb_image}")

### Launch the Spot Training Job

The training job will request spot capacity, train the model, and report cost savings in the output.

> **What is happening:** SageMaker requests a spot instance. Once capacity is available, training starts. The XGBoost built-in algorithm checkpoints automatically after each boosting round. When training completes, the output shows training seconds, billable seconds, and the managed spot savings percentage.

In [ ]:
print("Starting spot training job...")
print("This may take 3-7 minutes (includes waiting for spot capacity).")

spot_estimator.fit(
    {
        "train": TrainingInput(
            f"s3://{bucket}/{prefix}/data/xgb-train/", content_type="text/csv"
        ),
        "validation": TrainingInput(
            f"s3://{bucket}/{prefix}/data/xgb-validation/", content_type="text/csv"
        ),
    },
    wait=True,
    logs="All",
)

spot_model_uri = spot_estimator.model_data
print(f"\nSpot training complete.")
print(f"Model artifact: {spot_model_uri}")

training_job_name = spot_estimator.latest_training_job.name
job_desc = sm_client.describe_training_job(TrainingJobName=training_job_name)

training_seconds = job_desc.get("TrainingTimeInSeconds", 0)
billable_seconds = job_desc.get("BillableTimeInSeconds", 0)

if training_seconds > 0:
    savings_pct = (1 - billable_seconds / training_seconds) * 100
else:
    savings_pct = 0

print(f"\n--- Managed Spot Training Savings ---")
print(f"Training seconds:  {training_seconds}")
print(f"Billable seconds:  {billable_seconds}")
print(f"Managed Spot Savings: {savings_pct:.1f}%")

## Spot Instances for HPO -- Compound Savings

Thursday you ran HPO with multiple training jobs. Each job was an on-demand instance. Now imagine running those same jobs with spot instances.

**Why this works well:**
- Each HPO trial is an independent training job
- A spot interruption on one trial does not affect the others
- The tuner handles failed trials gracefully (marks them as failed, continues with the rest)
- Savings compound: 20 trials x 70% spot discount = significant cost reduction

**Cost example:**

| Scenario | Instance | Trials | On-Demand Cost | Spot Cost (70% savings) |
|----------|----------|--------|---------------|------------------------|
| Small HPO | ml.m5.xlarge | 10 | ~$4.50 | ~$1.35 |
| Medium HPO | ml.p3.2xlarge | 20 | ~$76.60 | ~$22.98 |
| Large HPO | ml.p3.8xlarge | 50 | ~$734.00 | ~$220.20 |

> **Discussion:** What happens if spot capacity is unavailable for an extended period during HPO? (Individual trials fail if they exceed `max_wait`. The tuner marks those trials as failed and continues. You may get fewer completed trials, but the best result from completed trials is still valid.)

In [ ]:
from sagemaker.tuner import (
    HyperparameterTuner,
    ContinuousParameter,
    IntegerParameter,
)

spot_estimator_for_hpo = sagemaker.estimator.Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/spot-hpo-output/",
    sagemaker_session=session,
    use_spot_instances=True,
    max_wait=3600,
    max_run=1800,
    checkpoint_s3_uri=f"s3://{bucket}/{prefix}/checkpoints/spot-hpo-{TIMESTAMP}",
)

spot_estimator_for_hpo.set_hyperparameters(
    objective="binary:logistic",
    num_round=100,
    eval_metric="auc",
)

hyperparameter_ranges = {
    "max_depth": IntegerParameter(3, 10),
    "eta": ContinuousParameter(0.01, 0.3),
    "subsample": ContinuousParameter(0.5, 1.0),
    "colsample_bytree": ContinuousParameter(0.5, 1.0),
}

tuner = HyperparameterTuner(
    estimator=spot_estimator_for_hpo,
    objective_metric_name="validation:auc",
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=10,
    max_parallel_jobs=2,
    strategy="Bayesian",
)

print("HPO Tuner configured with spot-enabled estimator:")
print(f"  max_jobs:          10")
print(f"  max_parallel_jobs: 2")
print(f"  strategy:          Bayesian")
print(f"  use_spot:          True")
print(f"  max_wait:          3600s")
print(f"  max_run:           1800s")
print()
print("NOTE: We are NOT launching this tuner to save time and cost.")
print("The configuration above is ready to use. To run it:")
print("  tuner.fit({'train': TrainingInput(...), 'validation': TrainingInput(...)})")

## Instance Right-Sizing Guide

Spot savings are powerful, but they compound when combined with right-sizing. If you train on an ml.p3.8xlarge when an ml.m5.xlarge would suffice, you are wasting money even with spot discounts.

| Family | Optimized For | Use When | Example Instance | On-Demand Cost (approx) |
|--------|--------------|----------|-----------------|------------------------|
| **ml.m5** | General purpose | Tabular data, XGBoost, small models, preprocessing | ml.m5.xlarge | ~$0.27/hr |
| **ml.c5** | Compute | CPU-intensive feature engineering, ensemble models, heavy preprocessing | ml.c5.2xlarge | ~$0.47/hr |
| **ml.p3** | GPU (training) | Deep learning training -- CNNs, RNNs, transformers, large models | ml.p3.2xlarge | ~$3.83/hr |
| **ml.g4dn** | GPU (inference) | GPU inference at lower cost than p3, small to medium DL models | ml.g4dn.xlarge | ~$0.74/hr |
| **ml.r5** | Memory | Large datasets that must fit in memory, embeddings, NLP tokenization | ml.r5.xlarge | ~$0.36/hr |

Notice the **14x cost difference** between ml.m5.xlarge and ml.p3.2xlarge. If your workload is tabular data with XGBoost, the GPU is literally idle -- you are paying 14x more for hardware you are not using.

## Instance Selection Decision Framework

Use this flowchart when choosing an instance type for training or inference:

```
Is the model a deep learning model (CNN, RNN, Transformer)?
  YES --> Is this training or inference?
            Training --> ml.p3 (or ml.p4d for very large models)
            Inference --> ml.g4dn (cheaper GPU, optimized for inference)
  NO  --> Does the data fit in memory on a general-purpose instance?
            NO  --> ml.r5 (memory-optimized)
            YES --> Is the workload CPU-bound (heavy preprocessing)?
                      YES --> ml.c5 (compute-optimized)
                      NO  --> ml.m5 (general purpose)
```

For FraudShield's XGBoost model, **ml.m5.xlarge** is the right choice. XGBoost does not use GPUs. The dataset fits in memory. An ml.p3.2xlarge would work but waste GPU resources.

> **Discussion:** FraudShield is adding a deep learning fraud detection model alongside XGBoost. It uses a transformer architecture trained on transaction sequences. Which instance would you choose for training? For inference?

---
# STAGE 2 -- Cost Optimization for Inference

**Connecting to Friday:** Friday you deployed models with four inference patterns. Each pattern runs on instances that cost money. Auto-scaling matches capacity to demand so you pay for what you use. Inference Recommender tells you which instance type gives the best price-performance ratio.

Training is a one-time expense per model version. Inference is a **continuous** expense -- your endpoint runs 24/7. A single ml.m5.xlarge endpoint costs about $200/month. Auto-scaling and right-sizing are the levers to control this cost.

## Auto-Scaling Concepts

SageMaker endpoint auto-scaling uses the **Application Auto Scaling** service. Two policy types:

| Policy Type | How It Works | Best For |
|-------------|-------------|----------|
| **Target Tracking** | Maintains a metric at a target value (like a thermostat) | Most workloads -- simple, effective |
| **Step Scaling** | Adds/removes instances in steps based on CloudWatch alarm thresholds | Fine-grained control, complex traffic patterns |

**Target tracking key metric:** `SageMakerVariantInvocationsPerInstance`
- Average number of invocations per instance per minute
- SageMaker scales out when the metric exceeds the target
- SageMaker scales in when the metric drops below the target

**Configuration parameters:**

| Parameter | Purpose | Example |
|-----------|---------|--------|
| `MinCapacity` | Minimum instances (never scale below) | 1 |
| `MaxCapacity` | Maximum instances (budget protection) | 4 |
| `TargetValue` | Target invocations per instance per minute | 100 |
| `ScaleInCooldown` | Seconds to wait before scaling in again | 300 |
| `ScaleOutCooldown` | Seconds to wait before scaling out again | 60 |

Think of target tracking like a thermostat. You set the temperature (target invocations per instance) and the system adjusts the heating (instance count) to maintain it.

## STEP 2 -- Configure Target Tracking Auto-Scaling

We register a SageMaker endpoint variant as a scalable target and attach a target tracking policy.

**Note:** We use a placeholder endpoint name for this demonstration. In production, you would use the name of your deployed endpoint.

> **What is happening:** We call the Application Auto Scaling API to register the endpoint variant as a scalable target (min 1, max 4 instances). Then we create a target tracking policy that targets 100 invocations per instance per minute. If traffic exceeds this, SageMaker adds instances. If it drops, instances are removed.

In [ ]:
aas_client = boto3.client("application-autoscaling")

ENDPOINT_NAME = f"fraudshield-autoscale-demo-{TIMESTAMP}"
VARIANT_NAME = "AllTraffic"

resource_id = f"endpoint/{ENDPOINT_NAME}/variant/{VARIANT_NAME}"

print("Registering scalable target...")
print(f"  Resource ID: {resource_id}")
print(f"  MinCapacity: 1")
print(f"  MaxCapacity: 4")
print()

try:
    aas_client.register_scalable_target(
        ServiceNamespace="sagemaker",
        ResourceId=resource_id,
        ScalableDimension="sagemaker:variant:DesiredInstanceCount",
        MinCapacity=1,
        MaxCapacity=4,
    )
    print("Scalable target registered successfully.")
except Exception as e:
    print(f"NOTE: Registration requires a live endpoint. Error: {e}")
    print("In production, this call succeeds after deploying an endpoint.")

print("\nCreating target tracking scaling policy...")
try:
    aas_client.put_scaling_policy(
        PolicyName=f"target-tracking-{TIMESTAMP}",
        ServiceNamespace="sagemaker",
        ResourceId=resource_id,
        ScalableDimension="sagemaker:variant:DesiredInstanceCount",
        PolicyType="TargetTrackingScaling",
        TargetTrackingScalingPolicyConfiguration={
            "TargetValue": 100.0,
            "PredefinedMetricSpecification": {
                "PredefinedMetricType": "SageMakerVariantInvocationsPerInstance",
            },
            "ScaleInCooldown": 300,
            "ScaleOutCooldown": 60,
        },
    )
    print("Target tracking policy created:")
    print("  TargetValue:      100 invocations/instance/minute")
    print("  ScaleInCooldown:  300 seconds")
    print("  ScaleOutCooldown: 60 seconds")
except Exception as e:
    print(f"NOTE: Policy creation requires a registered target. Error: {e}")
    print("In production, this call succeeds after registering the scalable target.")

## Step Scaling Alternative

Step scaling gives more granular control than target tracking. Instead of a single target, you define thresholds and the corresponding scaling action at each threshold.

**Example step scaling configuration:**

```
0-50 invocations/min   --> 1 instance (baseline)
50-150 invocations/min  --> add 1 instance
150-300 invocations/min --> add 2 instances
300+ invocations/min    --> add 3 instances
```

Step scaling requires a **CloudWatch alarm** as the trigger. You create the alarm on the `InvocationsPerInstance` metric, then define step adjustments that fire when the alarm transitions between states.

For most SageMaker workloads, **target tracking is sufficient and simpler**. Use step scaling when you have well-understood traffic patterns with distinct tiers, or when target tracking scales too aggressively or too conservatively.

In [ ]:
print("Step scaling policy configuration (conceptual):")
print()

step_scaling_config = {
    "PolicyName": f"step-scaling-{TIMESTAMP}",
    "ServiceNamespace": "sagemaker",
    "ResourceId": resource_id,
    "ScalableDimension": "sagemaker:variant:DesiredInstanceCount",
    "PolicyType": "StepScaling",
    "StepScalingPolicyConfiguration": {
        "AdjustmentType": "ChangeInCapacity",
        "StepAdjustments": [
            {
                "MetricIntervalLowerBound": 0,
                "MetricIntervalUpperBound": 100,
                "ScalingAdjustment": 1,
            },
            {
                "MetricIntervalLowerBound": 100,
                "MetricIntervalUpperBound": 250,
                "ScalingAdjustment": 2,
            },
            {
                "MetricIntervalLowerBound": 250,
                "ScalingAdjustment": 3,
            },
        ],
        "Cooldown": 120,
    },
}

print(json.dumps(step_scaling_config, indent=2))
print()
print("This policy adds instances in steps:")
print("  0-100 above threshold   --> add 1 instance")
print("  100-250 above threshold --> add 2 instances")
print("  250+ above threshold    --> add 3 instances")
print()
print("NOTE: Not applied to a live endpoint. This shows the API structure.")

## Inference Recommender Overview

You know you need auto-scaling. But **on which instance type?** Inference Recommender answers this question empirically.

**How it works:**
1. You provide a model artifact (in Model Registry or as a `model.tar.gz`)
2. You specify a sample payload for benchmarking
3. SageMaker deploys the model on multiple instance types
4. It runs benchmark invocations on each
5. It returns a ranked list of recommendations with latency, throughput, and cost

**Two modes:**

| Mode | What SageMaker Does | Duration | Use When |
|------|-------------------|----------|----------|
| **Default** | Picks instance types to benchmark automatically | 30-45 min | Broad exploration, unsure which instances to try |
| **Advanced** | You specify instance types and traffic pattern | 45-60 min | Narrow evaluation, comparing specific candidates |

> **Discussion:** When would you choose the GPU instance despite higher cost per invocation? (When latency is the primary constraint -- 4ms vs 12ms matters for real-time fraud scoring at checkout. Cost efficiency and latency are different optimization targets.)

In [ ]:
print("Inference Recommender API structure (conceptual):")
print()

recommender_config = {
    "JobName": f"fraudshield-recommender-{TIMESTAMP}",
    "JobType": "Default",
    "RoleArn": role,
    "InputConfig": {
        "ModelPackageVersionArn": "<model-package-arn-from-registry>",
        "JobDurationInSeconds": 1800,
    },
}

print(json.dumps(recommender_config, indent=2))
print()
print("NOTE: Not launching a recommender job (takes 30-45 minutes).")
print("Below is a sample results table from a typical XGBoost model:")
print()

results = pd.DataFrame({
    "Instance Type": ["ml.m4.xlarge", "ml.m5.xlarge", "ml.c5.large", "ml.c5.xlarge", "ml.g4dn.xlarge"],
    "Invocations/min": [600, 1200, 800, 1500, 2000],
    "Model Latency (ms)": [12, 8, 9, 6, 4],
    "Cost/hr": ["$0.134", "$0.269", "$0.119", "$0.238", "$0.736"],
    "Cost/1M Invocations": ["$3.72", "$3.74", "$2.48", "$2.64", "$6.13"],
})

print(results.to_string(index=False))
print()
print("For XGBoost, ml.c5.large offers the best cost per invocation.")
print("The GPU instance (ml.g4dn) has lowest latency but highest cost per")
print("invocation -- because XGBoost does not use the GPU.")

## CloudWatch Dashboards Overview

Auto-scaling and right-sizing require visibility. CloudWatch dashboards give you real-time metrics for your SageMaker endpoints.

**Key metrics for SageMaker endpoints:**

| Metric | What It Measures | Why It Matters |
|--------|-----------------|----------------|
| `Invocations` | Total invocations per period | Traffic volume |
| `ModelLatency` | Time the model takes to respond (microseconds) | Model performance |
| `OverheadLatency` | SageMaker overhead beyond model latency (microseconds) | Infrastructure health |
| `CPUUtilization` | CPU usage percentage | Right-sizing signal |
| `MemoryUtilization` | Memory usage percentage | Right-sizing signal |
| `InvocationsPerInstance` | Invocations per instance per minute | Auto-scaling signal |

A well-designed dashboard answers three questions at a glance:
1. **Is the endpoint healthy?** (Invocations, errors)
2. **Is it right-sized?** (CPU/memory utilization)
3. **Is auto-scaling working?** (InvocationsPerInstance vs target)

> **What is happening:** The code below creates a CloudWatch dashboard with widgets for key endpoint metrics. You would replace the endpoint name with your actual deployed endpoint.

In [ ]:
cw_client = boto3.client("cloudwatch")

DASHBOARD_NAME = f"FraudShield-Endpoint-Dashboard-{TIMESTAMP}"
MONITORED_ENDPOINT = ENDPOINT_NAME

dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0, "y": 0, "width": 12, "height": 6,
            "properties": {
                "title": "Invocations",
                "metrics": [
                    ["AWS/SageMaker", "Invocations",
                     "EndpointName", MONITORED_ENDPOINT,
                     "VariantName", VARIANT_NAME]
                ],
                "period": 60,
                "stat": "Sum",
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12, "y": 0, "width": 12, "height": 6,
            "properties": {
                "title": "Model Latency (microseconds)",
                "metrics": [
                    ["AWS/SageMaker", "ModelLatency",
                     "EndpointName", MONITORED_ENDPOINT,
                     "VariantName", VARIANT_NAME]
                ],
                "period": 60,
                "stat": "Average",
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0, "y": 6, "width": 12, "height": 6,
            "properties": {
                "title": "Overhead Latency (microseconds)",
                "metrics": [
                    ["AWS/SageMaker", "OverheadLatency",
                     "EndpointName", MONITORED_ENDPOINT,
                     "VariantName", VARIANT_NAME]
                ],
                "period": 60,
                "stat": "Average",
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12, "y": 6, "width": 12, "height": 6,
            "properties": {
                "title": "CPU Utilization (%)",
                "metrics": [
                    ["/aws/sagemaker/Endpoints", "CPUUtilization",
                     "EndpointName", MONITORED_ENDPOINT,
                     "VariantName", VARIANT_NAME]
                ],
                "period": 60,
                "stat": "Average",
                "region": region,
            },
        },
    ],
}

try:
    cw_client.put_dashboard(
        DashboardName=DASHBOARD_NAME,
        DashboardBody=json.dumps(dashboard_body),
    )
    print(f"CloudWatch dashboard created: {DASHBOARD_NAME}")
    print(f"View it at: https://{region}.console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={DASHBOARD_NAME}")
except Exception as e:
    print(f"Dashboard creation: {e}")

print("\nDashboard contains 4 widgets:")
print("  1. Invocations (Sum per minute)")
print("  2. Model Latency (Average, microseconds)")
print("  3. Overhead Latency (Average, microseconds)")
print("  4. CPU Utilization (Average, %)")

## Cost Comparison Across Inference Patterns

Friday you learned five inference patterns. Now compare them on cost. The right pattern can save thousands per month.

| Pattern | Billing Model | Idle Cost | Est. Monthly Cost (500 TPS avg) | Best For |
|---------|--------------|-----------|--------------------------------|----------|
| **Real-Time** | Per instance-hour | Full instance cost | ~$200-400 (1-2 ml.m5.xlarge) | Steady, latency-sensitive traffic |
| **Serverless** | Per ms of compute | $0 (scale to zero) | ~$50-150 (depends on invocation duration) | Bursty traffic, dev/test |
| **Async** | Per instance-hour | Full instance cost (unless custom scale-to-zero) | ~$200 (1 ml.m5.xlarge) | Large payloads, tolerates delay |
| **Batch Transform** | Per instance-hour (job duration only) | $0 (ephemeral) | ~$5-20 per job | Bulk scoring, scheduled runs |
| **Multi-Model** | Per instance-hour (shared) | Full instance cost (shared across models) | ~$100-200 (1 instance, many models) | Many similar models |

**Key insight:** Match the billing model to the traffic pattern. Steady traffic with low latency requirements? Real-time with auto-scaling. Bursty traffic with idle periods? Serverless. Bulk scoring? Batch transform. Many models with moderate traffic each? Multi-model.

The wrong pattern wastes money; the right pattern optimizes it.

> **Discussion:** FraudShield has steady traffic during business hours (8 AM - 8 PM) and near-zero traffic overnight. What combination of patterns would minimize cost? Consider the operational complexity of switching patterns vs the cost savings.

---
# STAGE 3 -- Security and Governance

**Connecting to the curriculum:** Cost optimization handles the "how much does it cost" question. Security handles the "who can access it and how is it protected" question. For FraudShield, we are processing financial transaction data. Regulators (PCI DSS, SOC 2, GDPR) require encryption at rest and in transit, network isolation, and audit trails.

A model that is cheap but insecure is a compliance violation waiting to happen.

## VPC Configuration for SageMaker

By default, SageMaker training jobs and endpoints run in an **AWS-managed VPC**. They can access the internet, S3, and other AWS services directly. For regulated workloads, this is not acceptable.

**Why VPC isolation matters:**
- Data does not traverse the public internet
- Network traffic is controlled by security groups and NACLs
- Meets compliance requirements for network segmentation
- Limits blast radius if credentials are compromised

**What you configure:**
- **Subnets:** Private subnets (no internet gateway route) where SageMaker runs workloads
- **Security groups:** Control inbound and outbound traffic at the instance level
- **VPC endpoints (PrivateLink):** Private connections to S3, ECR, CloudWatch, and SageMaker API

**Common mistake:** Putting SageMaker in a VPC without VPC endpoints. The training job starts, tries to pull the container image from ECR, cannot reach ECR because there is no internet access, and times out. Always configure VPC endpoints for S3, ECR, CloudWatch, and the SageMaker API.

> **Discussion:** What happens if a SageMaker training job in a private subnet tries to pull a container image from ECR without a VPC endpoint? (The request has no route to ECR. The job hangs during container download and eventually times out with a cryptic error.)

In [ ]:
print("VPC configuration for SageMaker training jobs:")
print()

vpc_training_config = {
    "subnets": ["subnet-abc123", "subnet-def456"],
    "security_group_ids": ["sg-789xyz"],
}

print("Estimator with VPC configuration:")
print("""
estimator = sagemaker.estimator.Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    subnets=["subnet-abc123", "subnet-def456"],
    security_group_ids=["sg-789xyz"],
    ...
)
""")

print("Model deployment with VPC configuration:")
print("""
model.deploy(
    instance_type="ml.m5.xlarge",
    initial_instance_count=1,
    vpc_config={
        "Subnets": ["subnet-abc123", "subnet-def456"],
        "SecurityGroupIds": ["sg-789xyz"],
    },
)
""")

print("Key considerations:")
print("  - Subnets must be private (no internet gateway route)")
print("  - Security groups control inbound/outbound traffic")
print("  - SageMaker needs VPC endpoints for S3, ECR, CloudWatch, SageMaker API")
print("  - Without VPC endpoints, jobs in private subnets cannot pull containers")

## PrivateLink -- Keep Traffic Off the Public Internet

PrivateLink creates a **private connection** between your VPC and AWS services. Instead of traffic going:

```
VPC --> Internet Gateway --> AWS Service
```

It goes:

```
VPC --> VPC Endpoint --> AWS Service (private network)
```

The traffic never touches the public internet.

**VPC endpoints needed for SageMaker in a private subnet:**

| Service | Endpoint Type | Purpose | Cost |
|---------|--------------|---------|------|
| S3 | Gateway | Read/write training data, model artifacts, checkpoints | Free |
| ECR (api + dkr) | Interface | Pull container images for training and inference | ~$0.01/hr/AZ |
| CloudWatch Logs | Interface | Write training logs and monitoring logs | ~$0.01/hr/AZ |
| SageMaker API | Interface | API calls (create training job, deploy endpoint) | ~$0.01/hr/AZ |
| SageMaker Runtime | Interface | Invoke endpoint (inference requests) | ~$0.01/hr/AZ |
| STS | Interface | Assume IAM roles | ~$0.01/hr/AZ |
| KMS | Interface | Encrypt/decrypt with customer-managed keys | ~$0.01/hr/AZ |

**Cost of network security:** For a production deployment with 6-7 interface endpoints across 2 AZs, that is roughly $100/month. Trivial compared to the compliance risk of routing financial data over the public internet.

> **Discussion:** Is $100/month for VPC endpoints worth it for FraudShield? (Absolutely. The alternative is processing financial data over the public internet, which is a PCI DSS violation.)

## KMS Encryption -- At Rest and In Transit

**Encryption at rest** protects data stored in S3, EBS volumes, and SageMaker storage. **Encryption in transit** protects data moving between services. Both are required for PCI DSS and SOC 2 compliance.

### What gets encrypted at rest:

| Storage Location | Default Encryption | Customer-Managed KMS |
|-----------------|-------------------|---------------------|
| S3 (training data, model artifacts) | SSE-S3 (AES-256) | SSE-KMS with custom key |
| EBS volumes (training instances) | AWS-managed key | Customer-managed KMS key |
| SageMaker storage (notebooks, endpoints) | AWS-managed key | Customer-managed KMS key |

**Default vs customer-managed keys:**
- Default encryption uses AWS-managed keys -- you get encryption but no control over key rotation, policies, or access logging.
- Customer-managed KMS keys give full control -- you define who can use the key, you set rotation schedules, and CloudTrail logs every key usage.

### Encryption in transit:
- All SageMaker API calls use TLS 1.2
- S3 transfers use HTTPS by default
- Inter-container traffic (distributed training) can be encrypted with `encrypt_inter_container_traffic=True`
- Inter-container encryption adds ~5-10% latency overhead

In [ ]:
print("KMS encryption configuration for SageMaker:")
print()

kms_key_id = "arn:aws:kms:us-east-1:123456789012:key/example-key-id"

print("Training job with KMS encryption:")
print(f"""
estimator = sagemaker.estimator.Estimator(
    image_uri=xgb_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_kms_key="{kms_key_id}",          # Encrypts model artifact in S3
    volume_kms_key="{kms_key_id}",           # Encrypts EBS volume on training instance
    encrypt_inter_container_traffic=True,    # Encrypts data between instances (distributed)
    ...
)
""")

print("Endpoint deployment with KMS encryption:")
print(f"""
model.deploy(
    instance_type="ml.m5.xlarge",
    initial_instance_count=1,
    kms_key="{kms_key_id}",                  # Encrypts endpoint storage volume
)
""")

print("S3 bucket encryption with KMS:")
print(f"""
s3.put_bucket_encryption(
    Bucket=bucket,
    ServerSideEncryptionConfiguration={{
        "Rules": [{{
            "ApplyServerSideEncryptionByDefault": {{
                "SSEAlgorithm": "aws:kms",
                "KMSMasterKeyID": "{kms_key_id}",
            }},
            "BucketKeyEnabled": True,
        }}],
    }},
)
""")

print("NOTE: Replace the example key ARN with your actual KMS key ARN.")
print("The SageMaker execution role needs kms:Decrypt and kms:GenerateDataKey")
print("permissions on the KMS key.")

## IAM Least-Privilege Review

The SageMaker execution role defines what SageMaker can do on your behalf. In development, broad roles are common. In production, **least-privilege is mandatory.**

### Key SageMaker IAM Permissions

| Permission | What It Allows | Least-Privilege Guidance |
|------------|---------------|------------------------|
| `sagemaker:CreateTrainingJob` | Start training jobs | Restrict to specific instance types and VPC configs |
| `sagemaker:CreateEndpoint` | Deploy endpoints | Restrict to specific endpoint config names |
| `sagemaker:InvokeEndpoint` | Call deployed models | Restrict to specific endpoint ARNs |
| `s3:GetObject` | Read from S3 | Restrict to specific bucket and prefix |
| `s3:PutObject` | Write to S3 | Restrict to specific output paths |
| `kms:Decrypt` / `kms:GenerateDataKey` | Use KMS keys | Restrict to specific key ARNs |
| `ecr:GetDownloadUrlForLayer` | Pull container images | Restrict to specific repositories |
| `logs:CreateLogGroup` / `PutLogEvents` | Write CloudWatch logs | Restrict to specific log groups |
| `iam:PassRole` | Pass the execution role to SageMaker | Restrict to specific role ARNs |

**The most dangerous permission is `iam:PassRole`.** If a user can pass any role to SageMaker, they can effectively escalate their own privileges. Always restrict `iam:PassRole` to specific role ARNs.

### Production IAM pattern:
- Separate roles for training, deployment, and monitoring
- Condition keys to restrict instance types (prevent accidental ml.p4d.24xlarge usage)
- Resource-level permissions (specific S3 paths, specific endpoints)
- Service control policies (SCPs) at the organization level as guardrails

> **Discussion:** What could go wrong if the SageMaker execution role has `s3:*` on all buckets? (The training job could read data from any bucket in the account, including buckets belonging to other teams. A bug or malicious code could exfiltrate data or overwrite critical files.)

## Full ML Lifecycle Review

We have spent two and a half weeks building an ML pipeline from end to end. This table maps every stage of the ML lifecycle to the SageMaker services and concepts we have covered.

| Lifecycle Stage | What Happens | SageMaker Service / Feature | When We Covered It |
|----------------|-------------|----------------------------|-------------------|
| **Prepare** | Collect, clean, transform data | Data Wrangler, Feature Store, Processing Jobs | W2 Tuesday |
| **Build** | Explore data, select features, choose algorithm | Studio Notebooks, Canvas (no-code), Autopilot (AutoML) | W2 Tuesday |
| **Train** | Train model on data | Estimator, Built-in Algorithms, Managed Spot Training | W1 Friday, W2 Thursday, W3 Monday |
| **Tune** | Optimize hyperparameters | HyperparameterTuner (Bayesian, Random, Grid, Hyperband) | W2 Thursday |
| **Evaluate** | Assess model performance | Processing Jobs, Clarify (bias, explainability) | W2 Monday, W2 Wednesday |
| **Register** | Version and catalog models | Model Registry, Model Groups, Approval Status | W2 Wednesday |
| **Approve** | Human or automated approval gate | Model Registry Approval Status, CI/CD integration | W2 Wednesday |
| **Deploy** | Serve model for inference | Real-Time, Serverless, Async, Batch Transform, Multi-Model | W2 Monday, W2 Friday |
| **Optimize** | Reduce cost, right-size, auto-scale | Spot Training, Instance Right-Sizing, Auto-Scaling, Inference Recommender | W3 Monday |
| **Secure** | Encrypt, isolate, restrict access | VPC, PrivateLink, KMS, IAM Least-Privilege | W3 Monday |
| **Monitor** | Detect drift, track performance | Model Monitor, Data Quality, Bias Drift, EventBridge Automation | W2 Friday |
| **Retrain** | Update model with new data | Pipelines, EventBridge triggers, Spot Training | W2 Monday, W2 Friday, W3 Monday |

Every stage connects to the next. The model you trained in Week 1 was deployed in Week 2, monitored in Week 2 Friday, and today you learned how to optimize and secure it. Tomorrow we shift from custom-trained models to foundation models and Amazon Bedrock -- a different paradigm, but the same lifecycle principles apply.

> **Discussion:** Which lifecycle stage is most often skipped in practice? Why? (Monitoring. Teams deploy and move on to the next project. The model degrades silently. This is why automating monitoring and alerting is critical.)

## Cleanup

Delete all resources created during this session: CloudWatch dashboards, auto-scaling policies, and any training artifacts.

> **What is happening:** We delete the CloudWatch dashboard and deregister any auto-scaling targets. The spot training job has already completed and released its instance. We verify no lingering resources remain.

In [ ]:
print("=== CLEANUP: Deleting all resources ===")
print()

print("1. Deleting CloudWatch dashboard...")
try:
    cw_client.delete_dashboards(DashboardNames=[DASHBOARD_NAME])
    print(f"   Deleted dashboard: {DASHBOARD_NAME}")
except Exception as e:
    print(f"   Dashboard: {e}")

print("\n2. Deregistering auto-scaling target...")
try:
    aas_client.deregister_scalable_target(
        ServiceNamespace="sagemaker",
        ResourceId=resource_id,
        ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    )
    print(f"   Deregistered scalable target: {resource_id}")
except Exception as e:
    print(f"   Scalable target: {e}")

print("\n3. Verifying no endpoints remain...")
try:
    endpoints = sm_client.list_endpoints(
        NameContains="fraudshield",
        StatusEquals="InService",
    )
    active = endpoints.get("Endpoints", [])
    if active:
        for ep in active:
            print(f"   WARNING: Active endpoint found: {ep['EndpointName']}")
            print(f"   Delete it manually in the AWS Console.")
    else:
        print("   No active fraudshield endpoints found.")
except Exception as e:
    print(f"   Endpoint check: {e}")

print("\n=== CLEANUP COMPLETE ===")
print("\nVERIFY MANUALLY:")
print("  1. Open AWS Console > SageMaker > Inference > Endpoints")
print("  2. Confirm no endpoints remain")
print("  3. Check Billing & Cost Management")

---
## Wrap-up

### What You Accomplished Today

1. **Cost Optimization for Training:** You configured Managed Spot Training to reduce training costs by up to 90%, set up S3-based checkpointing for fault tolerance, combined spot instances with HPO for compound savings, and learned a decision framework for selecting the right instance family (ml.m5, ml.c5, ml.p3, ml.g4dn, ml.r5).

2. **Cost Optimization for Inference:** You configured target tracking and step scaling auto-scaling policies to match capacity to demand, used Inference Recommender to benchmark instance selection, created a CloudWatch dashboard with key endpoint metrics, and compared cost profiles across all five inference patterns.

3. **Security and Governance:** You configured VPC isolation with private subnets and security groups, understood PrivateLink for keeping traffic off the public internet, configured KMS encryption at rest and in transit, reviewed IAM least-privilege principles, and mapped the full ML lifecycle from Prepare to Monitor across the entire curriculum.

### Tuesday Preview -- Amazon Bedrock

Tomorrow we shift paradigms. Instead of training our own models from scratch, we explore **Amazon Bedrock** and **foundation models**. The question changes from "how do we train a model" to "how do we leverage a pre-trained foundation model for our tasks." Read the Bedrock concept threads before Tuesday.

### Key Takeaways

- Managed Spot Training saves 60-90% on training compute. Checkpointing makes it practical.
- Right-sizing prevents paying for idle hardware. A 14x cost difference between ml.m5 and ml.p3 is real.
- Auto-scaling matches inference capacity to demand. Target tracking handles most workloads.
- VPC isolation, KMS encryption, and IAM least-privilege are non-negotiable for regulated industries.
- The full ML lifecycle (Prepare through Monitor) maps to SageMaker services covered across the entire curriculum.